# Spurs Tracker — Interactive Demo

This notebook walks through the `spurs_tracker` analytics suite.
Run `fetch.py` first to populate the cache, then explore below.

```bash
python ../fetch.py --team SAS --season 2024
```

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import plotly.io as pio
pio.renderers.default = 'notebook'

from config import TEAMS, season_str
from data.fetcher import Fetcher

TEAM_ABBR = 'SAS'
SEASON    = 2024

team_info = TEAMS[TEAM_ABBR]
team_id   = team_info['id']
team_name = team_info['name']
print(f'{team_name} — {season_str(SEASON)}')

In [ ]:
# Load cached data
fetcher     = Fetcher(team_id=team_id, season=SEASON)
games       = fetcher.load('games')
player_logs = fetcher.load('player_game_log')
roster      = fetcher.load('roster')

print(f'Games:       {len(games)}')
print(f'Player rows: {len(player_logs)}')
print(f'Roster:      {len(roster)}')
games.head()

## Season Record

In [ ]:
from analysis.standings import season_record, pythagorean_record, pace_projection

rec = season_record(games)
print(f"Record:       {rec['wins']}-{rec['losses']} ({rec['win_pct']:.3f})")
print(f"Net Rating:   {rec['net_rtg']:+.1f}")
print(f"Pythagorean:  {rec['pyth_wins']}-{rec['pyth_losses']}")
print(f"82-Game Pace: {pace_projection(rec['wins'], rec['losses'])}")

## Season Timeline Chart

In [ ]:
from viz.charts import season_timeline
season_timeline(games, team_name).show()

## Player Scoring Leaderboard

In [ ]:
from analysis.offense import scoring_leaderboard
scoring_leaderboard(player_logs, min_games=10)

## Hot / Cold Tracker

In [ ]:
from analysis.offense import hot_cold_summary
from viz.charts import hot_cold_chart

hc = hot_cold_summary(player_logs, windows=[5, 10], min_games=10)
hot_cold_chart(hc, team_name).show()

## Shot Efficiency (USG% vs TS%)

In [ ]:
from viz.charts import shot_efficiency_chart
shot_efficiency_chart(player_logs, team_name).show()

## Streak Analysis

In [ ]:
from analysis.streaks import current_streak, longest_streak, back_to_back_record

streak_t, streak_l = current_streak(games)
print(f'Current streak:  {streak_t}{streak_l}')
print(f'Longest W-streak: {longest_streak(games, "W")}')
print(f'Longest L-streak: {longest_streak(games, "L")}')

b2b = back_to_back_record(games)
print(f'\nBack-to-back record: {b2b["b2b_w"]}-{b2b["b2b_l"]}')
print(f'Normal rest record:  {b2b["rest_w"]}-{b2b["rest_l"]}')

## Full Dashboard (HTML)

In [ ]:
from viz.dashboard import build
path = build(games=games, player_logs=player_logs,
             team_name=team_name, team_abbr=TEAM_ABBR, season=SEASON)
print(f'Dashboard written to: {path}')